# 3단계-B: Multi-hop 문서 검색 (hop 1 → hop 2) (Colab, GPU)

베이스라인 문서 검색은 claim으로 **한 번** 검색한다. 2-hop의 두 번째 문서는 claim에 이름이 없는 경우가 많아서 TF-IDF top-20 안에 두 정답 문서가 다 들어오는 claim이 약 60%뿐이었다.
실패한 claim의 85%는 **문서 하나는 찾고 나머지 하나를 놓친** 경우다. 찾은 문서를 단서로 다음 문서를 찾는다.

```
hop 1:  3단계 재정렬 top-B 문서 (B=2)                    a
hop 2 후보: claim에 언급된 위키 제목 ∪ a 본문에 언급된 위키 제목   (위키 링크를 따라가는 것과 같은 원리)
hop 2 점수: roberta (claim | a 내용, 후보 c) → P(c가 나머지 정답 문서인가)
경로 점수: P(a) × P(c | a) → 높은 경로부터 (a, c)를 꺼내 문서 순위를 만든다
```

| 셀 | 역할 |
|---|---|
| 1 | 데이터, 3단계 재정렬 결과, 위키 DB |
| 2 | 위키 제목 사전 (약 480만 개), 본문/claim에서 제목 언급 찾기 |
| 3 | hop 2 후보로 정답 문서가 얼마나 들어오나 (학습 전 천장, val로 확인) |
| 4 | hop 2 학습 쌍 만들기 |
| 5 | hop 2 모델 학습 |
| 6 | val/test 경로 점수 → 새 문서 순위, recall@k 비교 |
| 7 | 저장 → 4단계(`06`)에서 `TAG = '_mh'`로 읽는다 |

In [37]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers datasets accelerate

import json, os, re, random, sqlite3, unicodedata, collections
from functools import lru_cache
import numpy as np, pandas as pd

ROOT = '/content/drive/MyDrive/fact-verification'
RAW, PROC = f'{ROOT}/data/raw', f'{ROOT}/data/processed'
WIKI = '/content/wiki_wo_links.db'
if not os.path.exists(WIKI):
    !cp "{RAW}/wiki_wo_links.db" {WIKI}
con = sqlite3.connect(WIKI)
nfc = lambda s: unicodedata.normalize('NFC', s)  # 정답 제목은 NFC, 위키 DB/TF-IDF 제목은 NFD라서 비교 전에 통일

dfs = {}
for n in ['train', 'val', 'test']:
    d = pd.read_json(f'{PROC}/hover2_{n}.jsonl', lines=True)
    r = pd.read_json(f'{PROC}/hover2_{n}_docret.jsonl', lines=True)
    d = d.merge(r, on='uid', validate='1:1')
    d['reranked'] = d.reranked.map(lambda ts: [nfc(t) for t in ts])
    d['gold_docs'] = d.supporting_facts.map(lambda sf: list(dict.fromkeys(nfc(t) for t, _ in sf)))
    dfs[n] = d
{n: len(d) for n, d in dfs.items()}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


{'train': 8166, 'val': 886, 'test': 1126}

In [38]:
# "Life Goes On (Fergie song)"처럼 괄호가 붙은 제목도 본문의 "Life Goes On"으로 찾을 수 있게 괄호를 뗀 소문자 이름으로 묶는다
base2titles = collections.defaultdict(list)
for (tid,) in con.execute('SELECT id FROM documents'):
    t = nfc(tid)
    base2titles[re.sub(r'\s*\(.*\)$', '', t).lower()].append(t)
print('위키 제목', len(base2titles))

TOK = re.compile(r"[\w][\w'.\-&$]*|[^\w\s]")
STOP = {'the', 'a', 'an', 'of', 'in', 'on', 'at', 'and', 'or', 'to', 'for', 'by', 'with', 'from', 'as', 'is', 'was',
        'he', 'she', 'it', 'his', 'her', 'its', 'this', 'that', 'january', 'february', 'march', 'april', 'may', 'june',
        'july', 'august', 'september', 'october', 'november', 'december'}

def mentioned_titles(s, max_n=8):
    # 대문자/숫자로 시작하는 1~8 단어 구간을 위키 제목과 대조한다 (고유명사만 노려 후보 수를 1/3로 줄임)
    toks = TOK.findall(s)
    out = set()
    for i in range(len(toks)):
        if not (toks[i][0].isupper() or toks[i][0].isdigit()):
            continue
        for n in range(1, min(max_n, len(toks) - i) + 1):
            span = re.sub(r" ([^\w\s])", r"\1", ' '.join(toks[i:i + n])).strip(" .,;:'\"").lower()
            if n == 1 and (span in STOP or len(span) < 3):
                continue
            out.update(base2titles.get(span, ()))
    return out

@lru_cache(maxsize=None)
def doc_text(title):
    row = con.execute('SELECT text FROM documents WHERE id = ?', (unicodedata.normalize('NFD', title),)).fetchone()
    return nfc(row[0]) if row else ''

@lru_cache(maxsize=None)
def doc_mentions(title):
    return frozenset(mentioned_titles(doc_text(title)))

def hop2_cands(claim, a):
    return (mentioned_titles(claim) | doc_mentions(a)) - {a}

sorted(hop2_cands('Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and Kristian Zahrtmann.', 'Kristian Zahrtmann'))[:20]

위키 제목 4796014


['1843',
 '1843 (magazine)',
 '1917',
 '19th century',
 '19th century (disambiguation)',
 'Age (album)',
 'Age (genus)',
 'Age (geology)',
 'Age (journal)',
 'Age (model theory)',
 'Age (song)',
 'Danish (name)',
 'Golden',
 'Golden (Brandon Beal song)',
 'Golden (Failure album)',
 'Golden (Jill Scott song)',
 'Golden (Kit Downes Trio album)',
 'Golden (Lady Antebellum album)',
 'Golden (Lady Antebellum song)',
 'Golden (Parade of Lights EP)']

In [39]:
B = 2  # hop 1로 쓸 재정렬 상위 문서 수

def coverage(d, B):
    hit, sizes = 0, []
    for r in d.itertuples():
        hop1 = r.reranked[:B]
        pool = set(hop1)
        for a in hop1:
            pool |= hop2_cands(r.claim, a)
        hit += set(r.gold_docs) <= pool; sizes.append(len(pool))
    return round(hit / len(d), 3), int(np.median(sizes))

for n in ['val', 'test']:
    d = dfs[n]
    base = np.mean([set(r.gold_docs) <= set(r.reranked[:20]) for r in d.itertuples()])
    print(n, f'베이스라인 top-20 천장 {base:.3f}', {f'B={b}': coverage(d, b) for b in [1, 2, 3]}, '(천장, 후보 수 중앙값)')

val 베이스라인 top-20 천장 0.677 {'B=1': (0.74, 123), 'B=2': (0.884, 212), 'B=3': (0.893, 297)} (천장, 후보 수 중앙값)
test 베이스라인 top-20 천장 0.602 {'B=1': (0.722, 132), 'B=2': (0.847, 214), 'B=3': (0.866, 304)} (천장, 후보 수 중앙값)


In [40]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'roberta-base'
MAX_LEN = 256
N_NEG = 7
tok = AutoTokenizer.from_pretrained(MODEL)

def make_train_pairs(d, seed=42):
    rng = random.Random(seed)
    rows = []
    for r in d.itertuples():
        for a in r.gold_docs:  # 정답 문서 하나를 hop 1로 두고, 나머지 정답 문서를 hop 2 양성으로
            others = [g for g in r.gold_docs if g != a]
            cands = sorted(hop2_cands(r.claim, a) - set(others))
            rows += [{'claim': r.claim, 'a': a, 'c': c, 'labels': 1} for c in others]  # 후보에 없어도 넣는다 (3단계와 같은 이유)
            rows += [{'claim': r.claim, 'a': a, 'c': c, 'labels': 0} for c in rng.sample(cands, min(N_NEG, len(cands)))]
    return pd.DataFrame(rows)

def encode(b):
    first = [f'{c} | {a}. {doc_text(a)}' for c, a in zip(b['claim'], b['a'])]
    second = [f'{c}. {doc_text(c)}' for c in b['c']]
    return tok(first, second, truncation=True, max_length=MAX_LEN)

train_pairs = make_train_pairs(dfs['train'])
train_ds = Dataset.from_pandas(train_pairs).map(encode, batched=True, remove_columns=['claim', 'a', 'c'])
len(train_pairs), round(float(train_pairs.labels.mean()), 3)

Map:   0%|          | 0/130308 [00:00<?, ? examples/s]

(130308, 0.125)

In [41]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(
    output_dir='/content/out/hop2',
    num_train_epochs=1, learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=256,
    warmup_steps=0.1, weight_decay=0.01, fp16=True, seed=42,
    save_strategy='no', logging_steps=100, disable_tqdm=True, report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=DataCollatorWithPadding(tok))
trainer.train()
# 양성 비율이 약 1/8이라 아무것도 못 배운 모델의 손실은 약 0.38이다. 그보다 충분히 내려가야 한다

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4757.23it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.4876', 'grad_norm': '2.868', 'learning_rate': '4.853e-06', 'epoch': '0.02455'}
{'loss': '0.28', 'grad_norm': '3.329', 'learning_rate': '9.755e-06', 'epoch': '0.0491'}
{'loss': '0.1224', 'grad_norm': '0.3332', 'learning_rate': '1.466e-05', 'epoch': '0.07366'}
{'loss': '0.1145', 'grad_norm': '12.55', 'learning_rate': '1.956e-05', 'epoch': '0.09821'}
{'loss': '0.1008', 'grad_norm': '29.3', 'learning_rate': '1.95e-05', 'epoch': '0.1228'}
{'loss': '0.09073', 'grad_norm': '0.2132', 'learning_rate': '1.896e-05', 'epoch': '0.1473'}
{'loss': '0.07468', 'grad_norm': '20.31', 'learning_rate': '1.841e-05', 'epoch': '0.1719'}
{'loss': '0.06505', 'grad_norm': '0.056', 'learning_rate': '1.787e-05', 'epoch': '0.1964'}
{'loss': '0.08187', 'grad_norm': '10.88', 'learning_rate': '1.732e-05', 'epoch': '0.221'}
{'loss': '0.06009', 'grad_norm': '4.399', 'learning_rate': '1.677e-05', 'epoch': '0.2455'}
{'loss': '0.0518', 'grad_norm': '0.08615', 'learning_rate': '1.623e-05', 'epoch': '0.2701'}
{'l

TrainOutput(global_step=4073, training_loss=0.06514572936254025, metrics={'train_runtime': 1476.445, 'train_samples_per_second': 88.258, 'train_steps_per_second': 2.759, 'train_loss': 0.06514572936254025, 'epoch': 1.0})

In [42]:
# 웹 데모(app.py)에서 쓰도록 학습한 모델을 Drive에 저장한다
SAVE = '/content/drive/MyDrive/fact-verification/models/hop2'
trainer.save_model(SAVE); tok.save_pretrained(SAVE)
!ls -lh {SAVE}

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.99s/it]


total 479M
-rw------- 1 root root  740 Oct 25 08:33 config.json
-rw------- 1 root root 476M Oct 25 08:33 model.safetensors
-rw------- 1 root root  388 Oct 25 08:33 tokenizer_config.json
-rw------- 1 root root 3.4M Oct 25 08:33 tokenizer.json
-rw------- 1 root root 5.1K Oct 25 08:33 training_args.bin


In [43]:
KS = [2, 5, 10, 20]

def recall_at_k(ranked, golds, k):
    return round(float(np.mean([set(g) <= set(r[:k]) for r, g in zip(ranked, golds)])), 3)

def multihop_rank(d):
    rows = [{'uid': r.uid, 'claim': r.claim, 'a': a, 's1': s1, 'c': c}
            for r in d.itertuples()
            for a, s1 in zip(r.reranked[:B], r.rerank_scores[:B])
            for c in hop2_cands(r.claim, a)]
    p = pd.DataFrame(rows)
    ds = Dataset.from_pandas(p[['claim', 'a', 'c']]).map(encode, batched=True, remove_columns=['claim', 'a', 'c'])
    p['s2'] = torch.softmax(torch.tensor(trainer.predict(ds).predictions), -1)[:, 1].numpy()
    p['path'] = p.s1 * p.s2
    paths = {uid: list(zip(g.a, g.c)) for uid, g in p.sort_values('path', ascending=False).groupby('uid', sort=False)}
    return [merge(paths.get(r.uid, []), r.reranked) for r in d.itertuples()]

def merge(paths, hop1, k=20):
    # 최고 경로의 (hop 1, hop 2) 문서를 맨 앞에 두고, 나머지는 hop 1 순위와 다음 경로들을 번갈아 채운다.
    # 경로 순서로만 채우면 hop 1 순위 3~20위의 정답 문서가 밀려나기 때문이다
    out = []
    def add(t):
        if t not in out and len(out) < k:
            out.append(t)
    for t in (paths[0] if paths else ()):
        add(t)
    rest = [t for pair in paths[1:] for t in pair]
    for i in range(max(len(hop1), len(rest))):
        if i < len(hop1): add(hop1[i])
        if i < len(rest): add(rest[i])
    return out

for n in ['val', 'test']:
    d = dfs[n]
    d['mh_ranked'] = multihop_rank(d)
    print(n)
    print('  hop 1만 (베이스라인)', {k: recall_at_k(d.reranked, d.gold_docs, k) for k in KS})
    print('  hop 1 → hop 2      ', {k: recall_at_k(d.mh_ranked, d.gold_docs, k) for k in KS})

Map:   0%|          | 0/262094 [00:00<?, ? examples/s]

val
  hop 1만 (베이스라인) {2: 0.6, 5: 0.665, 10: 0.677, 20: 0.677}
  hop 1 → hop 2       {2: 0.771, 5: 0.879, 10: 0.895, 20: 0.907}


Map:   0%|          | 0/334984 [00:00<?, ? examples/s]

test
  hop 1만 (베이스라인) {2: 0.511, 5: 0.577, 10: 0.599, 20: 0.602}
  hop 1 → hop 2       {2: 0.71, 5: 0.842, 10: 0.871, 20: 0.885}


In [44]:
# 4단계(06)가 읽는 형식 그대로 저장. train은 06에서 정답 문서를 넣어 학습하므로 기존 순위를 그대로 쓴다
dfs['train']['mh_ranked'] = dfs['train'].reranked
for n, d in dfs.items():
    d[['uid']].assign(reranked=d.mh_ranked).to_json(f'{PROC}/hover2_{n}_docret_mh.jsonl', orient='records', lines=True, force_ascii=False)
!ls -lh "{PROC}" 

total 61M
-rw------- 1 root root 883K Oct 25 06:20 hover2_test_docret.jsonl
-rw------- 1 root root 541K Oct 25 09:16 hover2_test_docret_mh.jsonl
-rw------- 1 root root 2.4M Oct 21 13:28 hover2_test_e2e_retrieved.jsonl
-rw------- 1 root root 2.4M Oct 25 08:05 hover2_test_e2e_retrieved_mh.jsonl
-rw------- 1 root root 768K Oct 21 06:22 hover2_test.jsonl
-rw------- 1 root root 2.1M Oct 21 12:55 hover2_test_sentret.jsonl
-rw------- 1 root root 2.1M Oct 25 07:57 hover2_test_sentret_mh.jsonl
-rw------- 1 root root 6.3M Oct 25 06:20 hover2_train_docret.jsonl
-rw------- 1 root root 4.2M Oct 25 09:16 hover2_train_docret_mh.jsonl
-rw------- 1 root root 5.5M Oct 21 06:22 hover2_train.jsonl
-rw------- 1 root root  15M Oct 21 12:55 hover2_train_sentret.jsonl
-rw------- 1 root root  15M Oct 25 07:57 hover2_train_sentret_mh.jsonl
-rw------- 1 root root 696K Oct 25 06:20 hover2_val_docret.jsonl
-rw------- 1 root root 427K Oct 25 09:16 hover2_val_docret_mh.jsonl
-rw------- 1 root root 612K Oct 21 06:22 